# Simple 1D‑CNN for Protein Function Prediction
This notebook trains a Convolutional Neural Network on the three pre‑processed datasets,
using PyTorch.

## Prerequisites
1. Run the **classical ML notebook** first (it saves `data/processed/X1.npy`, `y1.npy`, …).
2. Make sure the directory structure is present:
   - `data/processed/`
   - `outputs/models/`
   - `outputs/figures/`

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim
from copy import deepcopy
import time
import joblib

sns.set_style("whitegrid")

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

ensure_dir("outputs/models")
ensure_dir("outputs/figures")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Load Pre‑processed Data

In [ ]:
# Same arrays used in classical_ml.ipynb
X1 = np.load('data/processed/X1.npy')
y1_raw = np.load('data/processed/y1.npy')

X2 = np.load('data/processed/X2.npy')
y2_raw = np.load('data/processed/y2.npy')

X3 = np.load('data/processed/X3.npy')
y3_raw = np.load('data/processed/y3.npy')

print("Shapes:")
print(f"Dataset1: X={X1.shape}, y={y1_raw.shape}")
print(f"Dataset2: X={X2.shape}, y={y2_raw.shape}")
print(f"Dataset3: X={X3.shape}, y={y3_raw.shape}")

## 2. Define the 1D‑CNN Architecture

In [ ]:
class Protein1DCNN(nn.Module):
    def __init__(self, input_dim, num_classes, dropout=0.3):
        """
        input_dim : number of features (e.g., 27, 26, 29)
        num_classes : number of output classes
        """
        super().__init__()
        self.conv1 = nn.Conv1d(1, 64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(64)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(128)
        self.conv3 = nn.Conv1d(128, 256, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(256)
        self.pool = nn.AdaptiveMaxPool1d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        # x shape: (batch, features, 1) -> permute to (batch, 1, features)
        x = x.permute(0, 2, 1)
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.pool(x).squeeze(-1)  # -> (batch, 256)
        x = self.dropout(x)
        x = self.fc(x)
        return x

## 3. Training & Evaluation Helper

In [ ]:
def run_cnn_experiment(X, y_raw, dataset_name, output_prefix,
                       batch_size=64, lr=0.001, epochs=50, patience=10):
    # Encode labels
    le = LabelEncoder()
    y = le.fit_transform(y_raw)
    class_names = le.classes_
    num_classes = len(class_names)
    joblib.dump(le, f"outputs/models/{output_prefix}_cnn_label_encoder.pkl")

    # Split into train+val / test
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    # Further split train_val into train / val
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val, test_size=0.2, random_state=42, stratify=y_train_val
    )

    print(f"{dataset_name}: train={X_train.shape}, val={X_val.shape}, test={X_test.shape}, classes={num_classes}")

    # Scale features
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)

    # Reshape to 3D: (samples, features, 1)
    X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
    X_val = X_val.reshape(X_val.shape[0], X_val.shape[1], 1)
    X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

    # Convert to tensors
    train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                                  torch.tensor(y_train, dtype=torch.long))
    val_dataset = TensorDataset(torch.tensor(X_val, dtype=torch.float32),
                                torch.tensor(y_val, dtype=torch.long))
    test_dataset = TensorDataset(torch.tensor(X_test, dtype=torch.float32),
                                 torch.tensor(y_test, dtype=torch.long))

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # Model, loss, optimizer
    input_dim = X_train.shape[1]
    model = Protein1DCNN(input_dim, num_classes, dropout=0.3).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)

    # Training loop with early stopping
    best_val_loss = float('inf')
    best_model_state = None
    epochs_no_improve = 0
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)
        avg_train_loss = running_loss / len(train_dataset)
        history['train_loss'].append(avg_train_loss)

        # Validation
        model.eval()
        val_loss = 0.0
        correct = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                _, preds = torch.max(outputs, 1)
                correct += (preds == labels).sum().item()
        avg_val_loss = val_loss / len(val_dataset)
        val_acc = correct / len(val_dataset)
        history['val_loss'].append(avg_val_loss)
        history['val_acc'].append(val_acc)

        scheduler.step(avg_val_loss)

        print(f"Epoch {epoch+1}/{epochs} | train loss: {avg_train_loss:.4f} | val loss: {avg_val_loss:.4f} | val acc: {val_acc:.4f}")

        # Early stopping check
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_state = deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping after {epoch+1} epochs.")
                break

    # Load best model
    model.load_state_dict(best_model_state)

    # Plot training curves
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(history['train_loss'], label='Train Loss')
    ax1.plot(history['val_loss'], label='Val Loss')
    ax1.set_title(f'{dataset_name} - Loss')
    ax1.set_xlabel('Epoch')
    ax1.legend()
    ax2.plot(history['val_acc'], label='Val Accuracy', color='green')
    ax2.set_title(f'{dataset_name} - Val Accuracy')
    ax2.set_xlabel('Epoch')
    ax2.legend()
    plt.tight_layout()
    plt.savefig(f'outputs/figures/{output_prefix}_cnn_training.png', dpi=150)
    plt.show()

    # Test evaluation
    model.eval()
    y_pred_list = []
    with torch.no_grad():
        for inputs, _ in test_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            y_pred_list.extend(preds.cpu().numpy())
    y_pred = np.array(y_pred_list)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_test, y_pred, average='macro')
    f1 = f1_score(y_test, y_pred, average='macro')

    print(f"Test Accuracy: {acc:.4f} | F1: {f1:.4f}")

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=False, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'{dataset_name} - CNN Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    plt.savefig(f'outputs/figures/{output_prefix}_cnn_cm.png', dpi=150)
    plt.show()

    # Save model
    torch.save(model.state_dict(), f'outputs/models/{output_prefix}_cnn.pt')
    joblib.dump(scaler, f'outputs/models/{output_prefix}_cnn_scaler.pkl')

    # Return metrics as dict
    metrics = {
        'dataset': dataset_name,
        'model': 'CNN',
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'train_time': time.time()  # placeholder; can be improved
    }
    return metrics

## 4. Run CNN on Each Dataset

In [ ]:
all_cnn_results = []

# Dataset 1
all_cnn_results.append(
    run_cnn_experiment(X1, y1_raw, 'Dataset1', 'ds1', batch_size=64, epochs=50, patience=10)
)

# Dataset 2
all_cnn_results.append(
    run_cnn_experiment(X2, y2_raw, 'Dataset2', 'ds2', batch_size=64, epochs=50, patience=10)
)

# Dataset 3 (larger – may take time; reduce batch_size if needed)
all_cnn_results.append(
    run_cnn_experiment(X3, y3_raw, 'Dataset3', 'ds3', batch_size=128, epochs=50, patience=10)
)

## 5. Compile CNN Results

In [ ]:
cnn_df = pd.DataFrame(all_cnn_results)
cnn_df.to_csv('outputs/cnn_results.csv', index=False)
print("CNN results:")
cnn_df

In [ ]:
# Optional: Load classical model results and create combined comparison
if os.path.exists('outputs/results.csv'):
    classic_df = pd.read_csv('outputs/results.csv')
    all_df = pd.concat([classic_df, cnn_df], ignore_index=True)
    all_df.to_csv('outputs/all_results.csv', index=False)

    # Plot comparison
    metrics = ['accuracy', 'precision', 'recall', 'f1']
    melted = all_df.melt(id_vars=['dataset', 'model'], value_vars=metrics,
                         var_name='metric', value_name='score')
    plt.figure(figsize=(14, 7))
    sns.barplot(data=melted, x='metric', y='score', hue='model')
    plt.title('Performance Comparison: LR vs SVM vs CNN')
    plt.ylim(0, 1)
    plt.legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig('outputs/figures/all_models_comparison.png', dpi=150)
    plt.show()
else:
    print("Classical model results not found – run classical_ml.ipynb first.")

All CNN models, training curves, confusion matrices, and results have been saved to the `outputs/` folder.